In [ ]:
import torch

from models.autoencoder import pretrained
from utils.checkpoints import (
    load_ae_from_path,
    load_cspn_from_path,
    load_label_pc_from_path,
)
from utils.wandb_utils import load_from_wandb
from utils.visualisation import (
    plot_latent_space,
    show,
)


In [ ]:
ae_path = load_from_wandb("variational_celeba", "best")
ae = load_ae_from_path(ae_path, device="cpu")
# ae = pretrained.PretrainedVAE(name="stabilityai/sd-vae-ft-mse", width=64, height=64)

In [ ]:
cspn_path = load_from_wandb("psinet_celeba")
cspn = load_cspn_from_path(cspn_path, device="cpu")
cspn.eval()

In [ ]:
label_pc_path = load_from_wandb("label_pc_celeba")
label_pc = load_label_pc_from_path(label_pc_path, device="cpu")

In [ ]:
base = torch.zeros(40)
glasses = base.clone()
glasses[4] = 1

sample_labels = torch.stack([glasses] * 8).long()

with (torch.no_grad()):
    sampled_latents = cspn.sample(sample_labels)
    sampled_images = ae.decode(sampled_latents)

show(sampled_images, f"Samples from PSI-Net CSPN")

In [ ]:
from torchvision import datasets

dataset = datasets.CelebA(
    root="../data",
    split="test",
    download=True,
)

attributes = dataset.attr_names
attributes.remove("")

for i, attr in enumerate(attributes):
    with torch.no_grad():
        sample_labels_true = label_pc.complete_partial(known={i: 1}, batch_size=8).long()
        sampled_labels_false = label_pc.complete_partial(known={i: 0}, batch_size=8).long()
        sampled_latents_true = cspn.sample(sample_labels_true)
        sampled_latents_false = cspn.sample(sampled_labels_false)
        sampled_images_true = ae.decode(sampled_latents_true)
        sampled_images_false = ae.decode(sampled_latents_false)
    show(sampled_images_true, f"Samples from CSPN for attribute {attr} = 1", width=8)
    show(sampled_images_false, f"Samples from CSPN for attribute {attr} = 0", width=8)


In [ ]:
all_labels = torch.arange(2).repeat_interleave(4)

std_corrections = [0.0, 0.3, 0.5, 0.7, 1.0]

for std_correction in std_corrections:
    with torch.no_grad():
        samples_psi = cspn.sample(all_labels, std_correction=std_correction)
        sampled_images_psi_logits = ae.decode(samples_psi)
        sampled_images_psi = torch.sigmoid(sampled_images_psi_logits)

    show(
        sampled_images_psi,
        f"Samples from CSPN, std_correction={std_correction}",
        width=4,
    )

In [ ]:
sample_multi_labels = torch.arange(0, 2).repeat_interleave(1000)
with torch.no_grad():
    samples_psi = cspn.sample(sample_multi_labels)

plot_latent_space(samples_psi, sample_multi_labels)

In [ ]:
mpe_labels = torch.arange(0, 2)
with torch.no_grad():
    samples_psi = cspn.mpe(mpe_labels)
    sampled_images_psi_logits = ae.decode(samples_psi)
    sampled_images_psi = torch.sigmoid(sampled_images_psi_logits)

show(sampled_images_psi, "Samples from CSPN with MPE labels", width=10)

In [ ]:
num_samples = 16

with torch.no_grad():
    z_vae = torch.randn(num_samples, *ae.get_latent_dim())
    vae_images = ae.decode(z_vae)

    unconditional_labels = label_pc.complete_partial(
        known={}, batch_size=num_samples
    ).long()
    z_cspn = cspn.sample(unconditional_labels)
    cspn_images = ae.decode(z_cspn)

show(vae_images, "VAE samples", width=4)
show(cspn_images, "CSPN samples", width=4)